# Notebook 2 — Training and Evaluation

Trains a damage classifier on the patches prepared by notebook 1 and
evaluates it region by region against the PWTT baseline. **No Earth Engine
needed** — everything is read from Drive.

One experiment = one `CONFIG` dictionary. Changing the model, the features,
the split or the augmentations should never require edits below Section 1.
Every run writes its config, model checkpoint, metrics and figures to
`experiments/{experiment_name}/` on Drive, so runs can be compared by
comparing folders.

Two augmentations are available (both off by default, switch them in CONFIG):

* **Temporal label propagation** — a building marked damaged at one
  assessment date stays damaged at every later date, and before its first
  damaged label it counts as intact. With several assessment dates per city
  this multiplies the training labels. It needs those extra dates registered
  in `pipeline.py` and preprocessed by notebook 1.
* **Spatial smoothing** — at prediction time, each building's score is
  blended with its neighbours' scores, because damage is spatially
  clustered. Smoothing is applied *inside each region separately* (never
  across the train/test boundary), and the decision threshold is re-derived
  from smoothed validation scores so the metrics stay honest.

## 1. Setup and configuration

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
BASE = "/content/drive/MyDrive/War-Damage-Detection"
sys.path.append(BASE)

import os
import json
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (precision_recall_fscore_support, average_precision_score,
                             roc_auc_score, precision_recall_curve)

from pipeline import (PREP, CITY_REGISTRY, PWTT_THRESHOLD, PatchSource,
                      load_processed, select_channels, wanted_channels,
                      parse_split_entry, band_mask, split_assignment, processed_paths,
                      expand_labels_over_time, spatial_smooth,
                      MODEL_REGISTRY, build_model, predict_probs,
                      experiment_dirs)

CONFIG = {
    "experiment_name": "exp001_small_cnn_sar",
    "seed": 0,

    # which channels the model sees (must be a subset of what notebook 1
    # exported, see PREP["sensors"] in pipeline.py)
    "features": {
        "sentinel1": True,
        "sentinel2": False,
        "pre_event": True,
        "post_event": True,
    },

    # one of the keys in MODEL_REGISTRY: "small_cnn" or "siamese".
    # New models are added in pipeline.py with @register_model.
    "model": "small_cnn",

    # A split entry is "City", or "City:lo-hi" for a latitude quantile band,
    # optionally followed by "@date" to pin one assessment date.
    #
    # Bands make train, validation and test spatially disjoint: a random split
    # would leak through spatial autocorrelation, because a 32 px patch is
    # 320 m across and neighbouring buildings share pixels.
    #
    # The @date suffix adds a second, temporal question. Testing the SAME
    # held-out band at three dates asks whether the model still works as
    # damage accumulates and the city fills with rubble. Those buildings were
    # never trained on at any date, so this is not leakage - but the three
    # test rows are not independent of each other either, since they are the
    # same buildings seen later. Read them as a trend, not as three samples.
    #
    # Never hold out a DATE instead of a band: almost every building damaged
    # in May is still damaged in September, so training on one date and
    # testing on another would overlap almost completely.
    "split": {
        "train": ["Gaza:0.55-1.00"],              # all dates if label_temporal
        # the second-stage model needs CNN scores the CNN has NOT seen, so it
        # gets a band of its own rather than reusing train or val
        "stack": ["Gaza:0.42-0.55@20240503"],
        "val":   ["Gaza:0.33-0.42@20240503"],     # one date, so the threshold is fixed
        "test":  ["Gaza:0.00-0.33@20240503",
                  "Gaza:0.00-0.33@20240706",
                  "Gaza:0.00-0.33@20240906"],
    },

    # augmentations, see the notebook introduction
    "label_temporal": False,

    # Fixed-weight smoothing of the CNN's scores. Kept OFF: the base CNN
    # should judge each patch on its own, and the neighbourhood correction is
    # the second stage's job. Turn it on only to compare the hand-tuned
    # version against the learned one.
    "spatial_smoothing": {"enabled": False, "k": 8, "weight": 0.3},

    # Second stage: a gradient-boosted model over the CNN's scores and its
    # neighbours' scores. Where spatial_smoothing applies one global weight,
    # this learns when a neighbourhood should override a building's own score
    # and when it should not.
    "stacking": {
        "enabled": True,
        "ks": [8, 32],            # neighbourhood sizes, in buildings
        "n_estimators": 400,
        "max_depth": 4,
        "learning_rate": 0.05,
    },

    # training
    "batch_size": 128,
    "learning_rate": 3e-4,
    "max_epochs": 120,
    "early_stopping_patience": 10,
}


def validate_config(cfg):
    """Fail early with a readable message instead of a confusing crash later."""
    f = cfg["features"]
    if not (f["sentinel1"] or f["sentinel2"]):
        raise ValueError("enable at least one of sentinel1 / sentinel2")
    if not (f["pre_event"] or f["post_event"]):
        raise ValueError("enable at least one of pre_event / post_event")
    if cfg["model"] not in MODEL_REGISTRY:
        raise ValueError(f"unknown model, choose one of {list(MODEL_REGISTRY)}")
    if cfg["model"] == "siamese" and not (f["pre_event"] and f["post_event"]):
        raise ValueError("siamese compares pre and post, enable both")


validate_config(CONFIG)
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"

EXP_ROOT, OUT = experiment_dirs(CONFIG["experiment_name"])
with open(os.path.join(EXP_ROOT, "config.json"), "w") as fh:
    json.dump(CONFIG, fh, indent=2)

CHANNEL_NAMES = wanted_channels(CONFIG["features"])
print(f"experiment folder: {EXP_ROOT}")
print(f"device: {device}")
print(f"active channels ({len(CHANNEL_NAMES)}): {CHANNEL_NAMES}")

## 2. Build the train / validation / test sets

Each split entry becomes one "part": the patches, labels and coordinates of
one region. Test parts are kept separate so every region gets its own row in
the results table (Gaza-south measures within-city generalisation, the other
cities cross-conflict transfer).

If `label_temporal` is on, every **train** city with more than one registered
assessment date contributes one part per date, with labels propagated across
dates. Validation and test always use the latest date only.

In [ ]:
def make_part(d, name, lo, hi):
    """Cut one latitude band out of one loaded city."""
    m = band_mask(d["lat"], lo, hi)
    return {
        "name": name,
        "X": select_channels(d, CONFIG["features"]).rows(m),
        "y": d["y"][m],
        "xy": d["xy"][m],          # metres, used by spatial smoothing
        "lon": d["lon"][m],
        "lat": d["lat"][m],
        "gdf": d["gdf"].iloc[np.where(m)[0]],
    }


def parse_entry(entry):
    """'Gaza:0.00-0.33@20240906' -> ('Gaza', 0.0, 0.33, '20240906').

    The date is optional; None means "follow the temporal setting".
    """
    base, _, date = entry.partition("@")
    city, lo, hi = parse_split_entry(base)
    return city, lo, hi, (date or None)


def strip_dates(split_cfg):
    """The same split without @date suffixes, for functions that only map bands."""
    return {k: [e.split("@")[0] for e in v] for k, v in split_cfg.items()}


def build_parts(entries, temporal=False):
    parts = []
    for entry in entries:
        city, lo, hi, date = parse_entry(entry)
        dates = CITY_REGISTRY[city]["label_dates"]

        if date is not None:
            # an explicit date: use that assessment's own labels, untouched
            parts.append(make_part(load_processed(city, date), entry, lo, hi))
        elif temporal and len(dates) > 1:
            # one part per assessment date, with labels propagated forward
            frames = [(dt, load_processed(city, dt)) for dt in dates]
            new_y = expand_labels_over_time(frames)
            for dt, d in frames:
                d = dict(d)
                d["y"] = new_y[dt]
                parts.append(make_part(d, f"{entry}@{dt}", lo, hi))
        else:
            parts.append(make_part(load_processed(city), entry, lo, hi))
    return parts


train_parts = build_parts(CONFIG["split"]["train"],
                          temporal=CONFIG["label_temporal"])
val_parts = build_parts(CONFIG["split"]["val"])
stack_parts = (build_parts(CONFIG["split"]["stack"])
               if CONFIG["stacking"]["enabled"] and "stack" in CONFIG["split"] else [])
test_parts = build_parts(CONFIG["split"]["test"])

# PatchSource.concat glues the dates end to end, and .rows() above kept one
# latitude band - neither reads a pixel. The patches stay memory-mapped on
# disk, so only the batch being used is ever in RAM.
X_tr = PatchSource.concat([p["X"] for p in train_parts])
y_tr = np.concatenate([p["y"] for p in train_parts])
X_va = PatchSource.concat([p["X"] for p in val_parts])
y_va = np.concatenate([p["y"] for p in val_parts])

# Normalization statistics come from the training data and nowhere else.
# channel_stats streams the file in batches and accumulates in float64.
mu, sd = X_tr.channel_stats()

print(f"train n={len(y_tr):6d}  damaged={y_tr.mean()*100:5.2f} percent")
print(f"val   n={len(y_va):6d}  damaged={y_va.mean()*100:5.2f} percent")
for p in stack_parts:
    print(f"stack {p['name']:26s} n={len(p['y']):6d}  "
          f"damaged={p['y'].mean()*100:5.2f} percent")
for p in test_parts:
    print(f"test  {p['name']:26s} n={len(p['y']):6d}  "
          f"damaged={p['y'].mean()*100:5.2f} percent")

# the test bands must share no building with training, at any date
train_ids = set().union(*[set(p["gdf"]["system:index"]) for p in train_parts])
for p in test_parts + val_parts + stack_parts:
    overlap = train_ids & set(p["gdf"]["system:index"])
    if overlap:
        raise RuntimeError(f"{p['name']}: {len(overlap):,} buildings also appear "
                           f"in training. Check the band boundaries.")
print("\nno building appears in both training and evaluation")

### The split, on a map

`plot_split_map` draws every building coloured by the role its latitude band
gives it, with dashed lines at the band boundaries. The bands are **stacked,
not interleaved** — that is the whole point.

A 32 px patch is 320 m across, far wider than a building, so neighbouring
buildings share pixels. Under a random split those shared pixels would land on
both sides of the train/test divide and the test score would flatter the
model. Contiguous latitude bands make train, validation and test spatially
disjoint, so the only thing carrying over between them is what the model
actually learned.

The table underneath gives the size and damaged share of each part. Watch the
damaged shares: Gaza's damage is not uniform north to south, so the bands will
not be balanced, and a validation set with a very different prevalence from
the test set makes the threshold picked on it transfer badly.

In [ ]:
def plot_split_map(cfg, path=None):
    """Map every building by the role its latitude band gives it.

    The point of the figure is to make the split's spatial logic visible: the
    bands are stacked, not interleaved. A 32 px patch is 320 m across, far
    wider than a building, so neighbouring buildings share pixels. Under a
    random split those shared pixels would land on both sides of the
    train/test divide and the test score would flatter the model.

    Only coordinates and labels are read here, never the patches.
    """
    # the map shows WHERE each split is, so the @date suffixes are irrelevant
    # here and would confuse split_assignment
    bands = strip_dates(cfg["split"])
    cities = []
    for part in bands:
        for entry in bands[part]:
            city = parse_split_entry(entry)[0]
            if city not in cities:
                cities.append(city)

    colors = {"train": "#2E7D32", "stack": "#6A1B9A", "val": "#EF6C00",
              "test": "#C62828", "unused": "#CFCFCF"}

    # read coordinates first, so the figure can be shaped like the cities are
    data = {}
    for city in cities:
        npz_path, _, _ = processed_paths(city, CITY_REGISTRY[city]["label_dates"][-1])
        with np.load(npz_path, allow_pickle=False) as z:
            data[city] = (z["lat"], z["lon"], z["y"])

    # a degree of longitude is shorter than a degree of latitude away from the
    # equator, so set the aspect by cos(latitude) to get the real shape
    def extent(city):
        lat, lon, _ = data[city]
        k = np.cos(np.radians(lat.mean()))
        return (np.ptp(lon) * k), np.ptp(lat), 1 / k

    panel_w = 3.4
    tallest = max(h / (w or 1) for w, h, _ in map(extent, cities))
    fig, axes = plt.subplots(
        1, len(cities), squeeze=False,
        figsize=(panel_w * len(cities), min(panel_w * tallest + 1.0, 9.0)))

    rows = []
    for ax, city in zip(axes[0], cities):
        lat, lon, y = data[city]
        role = split_assignment(lat, city, bands)

        for r in ["unused", "train", "stack", "val", "test"]:   # grey first, so it sits behind
            m = role == r
            if not m.any():
                continue
            ax.scatter(lon[m], lat[m], s=1.0, alpha=0.4, color=colors[r],
                       label=r, linewidths=0)
            rows.append({"city": city, "split": r, "buildings": int(m.sum()),
                         "damaged %": round(y[m].mean() * 100, 2)})
            # label each band with its size and damaged share, in its own colour
            ax.text(0.98, lat[m].mean(), f"{r}\nn={m.sum():,}\n{y[m].mean()*100:.0f}% dmg",
                    transform=ax.get_yaxis_transform(), ha="right", va="center",
                    fontsize=8, color=colors[r], fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.3", fc="white",
                              ec=colors[r], alpha=0.85, lw=0.8))

        # dashed lines at the quantile boundaries that define the bands
        for part in bands:
            for entry in bands[part]:
                c_, lo, hi = parse_split_entry(entry)
                if c_ == city and (lo > 0 or hi < 1):
                    for q in (lo, hi):
                        ax.axhline(np.quantile(lat, q), color="black", lw=0.7,
                                   ls="--", alpha=0.55)

        ax.set_title(city, fontsize=11, pad=6)
        ax.set_aspect(extent(city)[2])
        ax.set_xticks([]); ax.set_yticks([])

    fig.suptitle("Train / validation / test split, by latitude band", fontsize=12)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()

    summary = pd.DataFrame(rows)
    summary = summary[summary["split"] != "unused"].reset_index(drop=True)
    print(summary.to_string(index=False))
    spread = summary["damaged %"].max() - summary["damaged %"].min()
    if spread > 15:
        print(f"\nNOTE: the damaged share varies by {spread:.0f} points across the "
              f"bands. Damage is not spread evenly north to south, so a threshold "
              f"picked on validation may transfer poorly to test. Compare F1 with "
              f"F1_best_thr in the results table to see how much of any gap that "
              f"explains.")
    return summary


split_summary = plot_split_map(CONFIG, os.path.join(OUT["figures"], "split_map.png"))

## 3. Training

One protocol for every architecture, so comparisons measure models rather
than training recipes: class-weighted loss for the imbalance, early stopping
on validation PR-AUC (under imbalance the loss can improve while ranking
quality falls), best checkpoint restored.

The decision threshold is chosen on **validation** scores — smoothed ones if
spatial smoothing is enabled, because smoothing changes the score
distribution and the threshold must match the scores it will be applied to.
Test data stays untouched until Section 4.

In [ ]:
class PatchDataset(Dataset):
    """Yields one (patch, label) pair. No augmentation is applied."""

    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        # read one patch from disk and normalize it here, so the float32 copy
        # never exists for more than a batch
        x = (np.asarray(self.X[i]).astype(np.float32) - mu[0]) / sd[0]
        return torch.from_numpy(x), torch.tensor(float(self.y[i]))


def score_parts(model, parts):
    """Predict probabilities for each part, smoothing per region if enabled.

    Smoothing runs inside one region at a time on purpose: blending scores
    across the train/test boundary would leak training-area information
    into test predictions.
    """
    sm = CONFIG["spatial_smoothing"]
    for p in parts:
        probs = predict_probs(model, p["X"], mu, sd, device=device)
        if sm["enabled"]:
            probs = spatial_smooth(p["xy"], probs, k=sm["k"], weight=sm["weight"])
        p["probs"] = probs
    return np.concatenate([p["probs"] for p in parts])


def train(cfg, model):
    tr_dl = DataLoader(PatchDataset(X_tr, y_tr), batch_size=cfg["batch_size"],
                       shuffle=True, drop_last=True)
    va_dl = DataLoader(PatchDataset(X_va, y_va), batch_size=256)

    # weight the rare class up so the loss does not ignore it
    pos_weight = torch.tensor([(len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1)],
                              dtype=torch.float32, device=device)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"],
                            weight_decay=1e-4)

    @torch.no_grad()
    def val_ap():
        model.eval()
        out = []
        for xb, _ in va_dl:
            out.append(torch.sigmoid(model(xb.to(device))).cpu())
        return average_precision_score(y_va, torch.cat(out).numpy())

    best_ap, best_epoch, best_state, bad, history = -np.inf, -1, None, 0, []
    for ep in range(1, cfg["max_epochs"] + 1):
        model.train()
        tot, seen = 0.0, 0
        for xb, yb in tr_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            tot += loss.item() * len(yb)
            seen += len(yb)

        ap = val_ap()
        history.append((ep, tot / seen, ap))
        star = ""
        if ap > best_ap + 1e-4:
            best_ap, best_epoch, bad = ap, ep, 0
            best_state = copy.deepcopy(model.state_dict())
            star = "  best so far"
        else:
            bad += 1
        print(f"epoch {ep:3d}  train loss {tot/seen:.4f}  val PR AUC {ap:.4f}{star}")
        if bad >= cfg["early_stopping_patience"]:
            print(f"early stop at epoch {ep}, best was epoch {best_epoch}")
            break

    model.load_state_dict(best_state)

    # threshold from validation scores - smoothed if smoothing is enabled,
    # so the threshold matches the scores it will later be applied to
    va_probs = score_parts(model, val_parts)
    prec, rec, thr = precision_recall_curve(y_va, va_probs)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    val_threshold = float(thr[np.argmax(f1)])

    ckpt_path = os.path.join(OUT["models"], f"{cfg['model']}.pt")
    torch.save({"state_dict": best_state, "mu": mu, "sd": sd,
                "channel_names": CHANNEL_NAMES, "config": cfg,
                "best_epoch": best_epoch, "val_ap": best_ap,
                "val_threshold": val_threshold}, ckpt_path)
    print(f"saved {ckpt_path}  (best epoch {best_epoch}, "
          f"val PR AUC {best_ap:.4f}, threshold {val_threshold:.3f})")
    return {"history": np.array(history), "best_epoch": best_epoch,
            "val_ap": best_ap, "val_threshold": val_threshold}


model = build_model(CONFIG["model"], CHANNEL_NAMES, device=device)
run = train(CONFIG, model)

model small_cnn: 24,017 parameters
epoch   1  train loss 0.2643  val PR AUC 0.8879  best so far
epoch   2  train loss 0.2453  val PR AUC 0.8869
epoch   3  train loss 0.2397  val PR AUC 0.8930  best so far
epoch   4  train loss 0.2370  val PR AUC 0.8777
epoch   5  train loss 0.2340  val PR AUC 0.8896


## 4. Evaluation

Every test region separately, CNN against the PWTT `max_change` baseline at
its published threshold. Besides the F1 at the validation threshold, the
table also reports the F1 at each region's *own* best threshold
(`F1_best_thr`): the gap between the two separates "the model does not
transfer" from "the threshold does not transfer", which are very different
findings.

Label files built by notebook 0 have no `max_change` column, since the PWTT
statistic has to be computed from imagery. Those regions simply get no PWTT
row and no dashed baseline curve.

In [ ]:
def binary_metrics(y_true, score, thresh):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, (score >= thresh).astype(int), average="binary", zero_division=0)
    # F1 at the best threshold FOR THIS REGION (diagnostic only - choosing
    # the threshold on test data is not a deployable procedure)
    prec, rec, thr = precision_recall_curve(y_true, score)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    return {"P": p, "R": r, "F1": f1, "F1_best_thr": float(f1s.max()),
            "PR_AUC": average_precision_score(y_true, score),
            "ROC_AUC": roc_auc_score(y_true, score),
            "prevalence": float(np.mean(y_true)), "n": len(y_true)}


score_parts(model, test_parts)   # fills p["probs"], smoothed per region if enabled

def pwtt_scores(part):
    """The PWTT statistic, or None if it was never computed for these rows.

    A handful of buildings can legitimately have no value - they sit where the
    T-statistic raster is nodata - so those are filled with a score below the
    threshold rather than throwing the whole baseline away. Only an entirely
    empty column means the baseline is genuinely missing (run Section 3b of
    notebook 1 to compute it).
    """
    if "max_change" not in part["gdf"].columns:
        return None
    mc = part["gdf"]["max_change"].to_numpy(float)
    if not np.isfinite(mc).any():
        return None
    missing = ~np.isfinite(mc)
    if missing.any():
        mc = np.where(missing, np.nanmin(mc), mc)
    return mc


rows = {}
for p in test_parts:
    rows[f"{p['name']}  CNN"] = binary_metrics(p["y"], p["probs"],
                                               run["val_threshold"])
    mc = pwtt_scores(p)
    if mc is None:
        print(f"{p['name']}: no PWTT statistic in the labels, baseline skipped")
    else:
        rows[f"{p['name']}  PWTT"] = binary_metrics(p["y"], mc, PWTT_THRESHOLD)

results = pd.DataFrame(rows).T
results.round(4).to_csv(os.path.join(OUT["metrics"], "test_metrics.csv"))
with open(os.path.join(OUT["metrics"], "run_summary.json"), "w") as fh:
    json.dump({"best_epoch": run["best_epoch"], "val_ap": run["val_ap"],
               "val_threshold": run["val_threshold"]}, fh, indent=2)
results.round(4)

## 5. Figures

Training curve, precision–recall curves per region, and a spatial error map. All saved to the experiment folder.

In [ ]:
def plot_training(history, path):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(history[:, 0], history[:, 1])
    ax[0].set_xlabel("epoch")
    ax[0].set_title("train loss")
    ax[1].plot(history[:, 0], history[:, 2], color="C2")
    ax[1].axhline(y_va.mean(), color="gray", ls=":",
                  label="chance level (prevalence)")
    ax[1].set_xlabel("epoch")
    ax[1].set_title("validation PR AUC")
    ax[1].legend()
    plt.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()


def plot_pr_curves(parts, path):
    fig, axes = plt.subplots(1, len(parts), figsize=(5.2 * len(parts), 4.4))
    axes = np.atleast_1d(axes)
    for ax, p in zip(axes, parts):
        pr, rc, _ = precision_recall_curve(p["y"], p["probs"])
        ax.plot(rc, pr,
                label=f"CNN AP={average_precision_score(p['y'], p['probs']):.3f}")
        mc = pwtt_scores(p)
        if mc is not None:
            pr2, rc2, _ = precision_recall_curve(p["y"], mc)
            ax.plot(rc2, pr2, "--",
                    label=f"PWTT AP={average_precision_score(p['y'], mc):.3f}")
        ax.axhline(p["y"].mean(), color="gray", ls=":")
        ax.set_ylim(0, 1)
        ax.set_title(f"{p['name']}  ({p['y'].mean()*100:.1f} percent damaged)")
        ax.set_xlabel("recall")
        ax.legend(fontsize=8)
    axes[0].set_ylabel("precision")
    plt.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()


def plot_error_map(part, threshold, path):
    pred = (part["probs"] >= threshold).astype(int)
    wrong = pred != part["y"]
    fig, ax = plt.subplots(figsize=(7, 8))
    sc = ax.scatter(part["lon"], part["lat"], c=part["probs"], s=5,
                    cmap="RdYlGn_r", vmin=0, vmax=1)
    ax.scatter(part["lon"][wrong], part["lat"][wrong], facecolors="none",
               edgecolors="black", s=26, linewidths=0.6, label="misclassified")
    plt.colorbar(sc, ax=ax, label="predicted damage probability")
    ax.set_title(f"{part['name']}  errors at threshold {threshold:.2f}")
    ax.set_aspect("equal")
    ax.legend(loc="lower right")
    plt.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()


plot_training(run["history"], os.path.join(OUT["figures"], "training.png"))
plot_pr_curves(test_parts, os.path.join(OUT["figures"], "pr_curves.png"))
plot_error_map(test_parts[0], run["val_threshold"],
               os.path.join(OUT["figures"], "error_map.png"))

### Does it hold up as the war goes on?

The three test rows are the same held-out band at three assessment dates. The
buildings never change, so anything that moves is the imagery, the labels, or
the model's fitness for a city that is progressively more damaged.

This is a fair test — those buildings are outside the training band at every
date — but the three rows are **not independent of each other**, since a
building damaged in May is still damaged in September. Read them as a trend
line, not as three separate experiments.

In [ ]:
def plot_metrics_over_time(parts, threshold, path=None):
    """How the model holds up as damage accumulates.

    Every point is the SAME held-out band scored at a different assessment
    date, so the buildings are identical and only the imagery and labels move.
    Two things to read off it:

    PR AUC against prevalence. PR AUC rises automatically as more of the city
    is damaged, so the gap to the dashed prevalence line is what matters, not
    the level. ROC AUC is prevalence-insensitive and is the fairer trend line.

    F1 against F1 at the best threshold. The threshold was fixed once on the
    validation band. If the two curves separate over time, the model is still
    ranking buildings well and only the cutoff has gone stale - a calibration
    problem, not a capability one.
    """
    dated = [p for p in parts if "@" in p["name"]]
    if len(dated) < 2:
        print("fewer than two dated test parts, nothing to plot over time")
        return None

    rows = []
    for p in sorted(dated, key=lambda q: q["name"].split("@")[1]):
        m = binary_metrics(p["y"], p["probs"], threshold)
        m["date"] = p["name"].split("@")[1]
        rows.append(m)
    df = pd.DataFrame(rows).set_index("date")

    x = pd.to_datetime(df.index)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))

    ax[0].plot(x, df["PR_AUC"], marker="o", label="PR AUC")
    ax[0].plot(x, df["ROC_AUC"], marker="s", label="ROC AUC")
    ax[0].plot(x, df["prevalence"], ls=":", color="gray", label="prevalence")
    ax[0].set_ylim(0, 1)
    ax[0].set_title("ranking quality")
    ax[0].legend(fontsize=8)

    ax[1].plot(x, df["F1"], marker="o", label=f"F1 at the fixed {threshold:.2f}")
    ax[1].plot(x, df["F1_best_thr"], marker="s", ls="--",
               label="F1 at this date's best threshold")
    ax[1].set_ylim(0, 1)
    ax[1].set_title("decision quality")
    ax[1].legend(fontsize=8)

    for a in ax:
        a.tick_params(axis="x", rotation=45)
    fig.suptitle("Held-out band, scored at each assessment date", y=1.02)
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()

    print(df[["n", "prevalence", "PR_AUC", "ROC_AUC", "F1", "F1_best_thr"]]
          .round(4).to_string())
    gap = (df["F1_best_thr"] - df["F1"]).max()
    if gap > 0.05:
        print(f"\nThe fixed threshold costs up to {gap:.3f} F1 at some dates. That "
              f"is calibration drift, not the model getting worse: prevalence in "
              f"this band moves from {df['prevalence'].iloc[0]:.2f} to "
              f"{df['prevalence'].iloc[-1]:.2f}, and the best cutoff moves with it.")
    return df


over_time = plot_metrics_over_time(
    test_parts, run["val_threshold"],
    os.path.join(OUT["figures"], "metrics_over_time.png"))

## 6. What does the model use?

**Modality ablation** zeroes one channel group at a time and measures the
PR-AUC drop — the honest answer to which input matters. **Grad-CAM** shows
which pixels of a patch drove one prediction, useful for checking that the
model looks at the building rather than an artifact.

In [ ]:
def modality_ablation(part):
    """PR AUC after zeroing each channel group. A big drop = the group matters."""
    groups = {}
    for i, name in enumerate(CHANNEL_NAMES):
        key = name.split("_")[0] + " " + ("pre" if "_pre_" in name else "post")
        groups.setdefault(key, []).append(i)
    rows = {"full input": average_precision_score(
        part["y"], predict_probs(model, part["X"], mu, sd, device=device))}
    for key, idx in groups.items():
        # zeroing happens inside predict_probs, per batch, instead of copying
        rows[f"without {key}"] = average_precision_score(
            part["y"], predict_probs(model, part["X"], mu, sd, device=device,
                                     zero_channels=idx))
    return pd.Series(rows, name="PR_AUC")


print(modality_ablation(test_parts[0]).round(4))

In [ ]:
import torch.nn.functional as F


def grad_cam(model, x):
    """Which pixels drove this prediction? (from the last conv layer)"""
    convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
    acts, grads = {}, {}
    h1 = convs[-1].register_forward_hook(
        lambda m, i, o: acts.__setitem__("v", o.detach()))
    h2 = convs[-1].register_full_backward_hook(
        lambda m, gi, go: grads.__setitem__("v", go[0].detach()))
    model.eval()
    xb = torch.from_numpy(x[None]).to(device).requires_grad_(True)
    logit = model(xb)
    model.zero_grad()
    logit.backward()
    h1.remove(); h2.remove()

    weight = grads["v"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weight * acts["v"]).sum(dim=1))[0].cpu().numpy()
    cam = cam - cam.min()
    if cam.max() > 0:
        cam = cam / cam.max()
    return cam, float(torch.sigmoid(logit).item())


def show_grad_cam_examples(part, n=4, path=None):
    """Grad-CAM for the highest and lowest scoring buildings of a region."""
    order = np.argsort(-part["probs"])
    picks = list(order[:n // 2]) + list(order[-(n - n // 2):])
    fig, axes = plt.subplots(2, len(picks), figsize=(3.2 * len(picks), 6.4))
    for col, j in enumerate(picks):
        xj = (np.asarray(part["X"][j]).astype(np.float32) - mu[0]) / sd[0]
        cam, prob = grad_cam(model, xj)
        cam_big = np.kron(cam, np.ones((xj.shape[1] // cam.shape[0],
                                        xj.shape[2] // cam.shape[1])))
        base = xj[min(2, xj.shape[0] - 1)]
        axes[0, col].imshow(base, cmap="gray")
        axes[0, col].set_title(f"p={prob:.2f}  y={part['y'][j]}", fontsize=9)
        axes[1, col].imshow(base, cmap="gray")
        axes[1, col].imshow(cam_big, cmap="jet", alpha=0.45)
        for ax in (axes[0, col], axes[1, col]):
            ax.set_xticks([]); ax.set_yticks([])
    axes[0, 0].set_ylabel("input")
    axes[1, 0].set_ylabel("Grad-CAM")
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=150)
    plt.show()


show_grad_cam_examples(test_parts[0], n=4,
                       path=os.path.join(OUT["figures"], "grad_cam.png"))

## 7. A second stage over the CNN's scores

The CNN judges each patch on its own — SAR pre and post, nothing else. That is
deliberate: it keeps the first stage a clean image classifier and leaves the
question of *what the neighbourhood implies* to a model that can actually
weigh it.

`spatial_smoothing` already does a crude version of this, blending each score
with its neighbours' at one fixed weight. But 0.3 is a guess, and the right
correction is not constant: a 0.6 in a block of 0.9s should probably move up,
while a 0.6 in a block of 0.1s might be a real isolated strike and should not
move at all. One global weight cannot express that.

So the second stage gets features and decides for itself:

* the building's own CNN score
* its 8 nearest neighbours' scores individually, sorted strongest first
* mean, standard deviation, min and max of the neighbourhood at two scales
  (8 buildings ≈ its own block, 32 ≈ the wider district)
* how far its score sits from the neighbourhood mean
* mean distance to those neighbours, which is a proxy for how dense the area is

The **standard deviation** is the interesting one. A low spread means the
neighbourhood agrees and the consensus is worth trusting; a high spread means
the block is mixed and the building's own score should carry more weight.

**Where it is trained matters more than the model choice.** On the training
band the CNN has effectively memorised the labels, so its scores there are far
cleaner and more confident than on unseen buildings. A stacker fitted to that
would learn to trust the CNN more than it deserves. It therefore gets its own
band, `Gaza:0.42-0.55`, carved out of what used to be validation.

In [ ]:
from pipeline import neighbour_features
from xgboost import XGBClassifier


def score_raw(parts):
    """CNN probabilities with no smoothing applied.

    score_parts() would apply spatial_smoothing if it were enabled, and the
    second stage must see the CNN's own opinion of each patch - correcting it
    twice would double count the neighbourhood.
    """
    for p in parts:
        p["probs"] = predict_probs(model, p["X"], mu, sd)


def build_stack_table(parts, ks):
    """Neighbour features for a set of parts, one region at a time.

    Neighbours are found INSIDE each part, never across parts. Pooling first
    and searching afterwards would let a test building take its neighbours
    from the training band, which is the leak the latitude split exists to
    prevent.
    """
    tables, labels = [], []
    for p in parts:
        tables.append(neighbour_features(p["xy"], p["probs"], ks=tuple(ks)))
        labels.append(p["y"])
    return pd.concat(tables, ignore_index=True), np.concatenate(labels)


def train_stacker(cfg, stack_parts, val_parts):
    """Fit the second stage on CNN scores it has never seen.

    Fitting on the training band instead would be a trap: there the CNN has
    effectively memorised the labels, so its scores are far cleaner and far
    more confident than on unseen buildings. A model tuned to that pattern
    learns to trust the CNN more than it should.
    """
    sc = cfg["stacking"]
    score_raw(stack_parts)
    score_raw(val_parts)
    X_stack, y_stack = build_stack_table(stack_parts, sc["ks"])
    X_val, y_val = build_stack_table(val_parts, sc["ks"])

    clf = XGBClassifier(
        n_estimators=sc["n_estimators"], max_depth=sc["max_depth"],
        learning_rate=sc["learning_rate"], subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr", early_stopping_rounds=40,
        scale_pos_weight=(y_stack == 0).sum() / max((y_stack == 1).sum(), 1),
        random_state=cfg["seed"], n_jobs=4)
    clf.fit(X_stack, y_stack, eval_set=[(X_val, y_val)], verbose=False)

    print(f"second stage: {len(X_stack):,} buildings, {X_stack.shape[1]} features, "
          f"best iteration {clf.best_iteration}")

    # the threshold is chosen on validation, exactly as for the CNN
    val_p = clf.predict_proba(X_val)[:, 1]
    prec, rec, thr = precision_recall_curve(y_val, val_p)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    threshold = float(thr[np.argmax(f1)])
    print(f"   validation PR AUC {average_precision_score(y_val, val_p):.4f}, "
          f"threshold {threshold:.3f}")

    imp = (pd.Series(clf.feature_importances_, index=X_stack.columns)
             .sort_values(ascending=False))
    print("\n   most useful features:")
    print(imp.head(8).round(4).to_string())
    return clf, threshold, imp


if not CONFIG["stacking"]["enabled"] or not stack_parts:
    print("stacking disabled, or no stack band in the split")
    stacker = None
else:
    stacker, stack_threshold, stack_importance = train_stacker(
        CONFIG, stack_parts, val_parts)

    # score the test regions, again one region at a time
    score_raw(test_parts)
    for p in test_parts:
        feats = neighbour_features(p["xy"], p["probs"],
                                   ks=tuple(CONFIG["stacking"]["ks"]))
        p["stack_probs"] = stacker.predict_proba(feats)[:, 1]

    torch.save({"model": stacker, "threshold": stack_threshold,
                "ks": CONFIG["stacking"]["ks"]},
               os.path.join(OUT["models"], "stacker.pt"))

### Is the extra stage earning its keep?

Three rows per region: the CNN alone, the CNN with fixed-weight smoothing, and
the CNN with the learned stage. The middle row is the control that matters — if
the stacker only matches it, the gain was just "average with your neighbours"
and the extra machinery is not worth the complexity. It has to beat the
hand-tuned version.

Watch PR AUC rather than F1 here, since all three use thresholds picked on
different score distributions.

In [ ]:
def compare_stages(parts, cnn_threshold, stack_threshold, path=None):
    """CNN alone, CNN plus fixed smoothing, and the learned second stage.

    The middle row is the fair control. If the second stage only matches it,
    the gain was just "average with your neighbours" and the extra machinery
    buys nothing. It has to beat the hand-tuned version to be worth keeping.
    """
    sm = CONFIG["spatial_smoothing"]
    rows = {}
    for p in parts:
        rows[f"{p['name']}  1 CNN"] = binary_metrics(p["y"], p["probs"], cnn_threshold)

        smoothed = spatial_smooth(p["xy"], p["probs"],
                                  k=sm["k"], weight=sm["weight"])
        rows[f"{p['name']}  2 CNN + smoothing"] = binary_metrics(
            p["y"], smoothed, cnn_threshold)

        if "stack_probs" in p:
            rows[f"{p['name']}  3 CNN + stacker"] = binary_metrics(
                p["y"], p["stack_probs"], stack_threshold)

    table = pd.DataFrame(rows).T
    table.round(4).to_csv(os.path.join(OUT["metrics"], "stage_comparison.csv"))
    print(table[["PR_AUC", "ROC_AUC", "F1", "F1_best_thr", "prevalence"]].round(4)
               .to_string())

    # PR curves for the first test region, one line per stage
    p = parts[0]
    sm_probs = spatial_smooth(p["xy"], p["probs"], k=sm["k"], weight=sm["weight"])
    series = [("CNN", p["probs"]), ("CNN + smoothing", sm_probs)]
    if "stack_probs" in p:
        series.append(("CNN + stacker", p["stack_probs"]))

    fig, ax = plt.subplots(figsize=(5.5, 4.4))
    for name, s in series:
        pr, rc, _ = precision_recall_curve(p["y"], s)
        ax.plot(rc, pr, label=f"{name}  AP={average_precision_score(p['y'], s):.3f}")
    ax.axhline(p["y"].mean(), color="gray", ls=":", label="chance")
    ax.set_xlabel("recall"); ax.set_ylabel("precision")
    ax.set_title(p["name"])
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return table


if stacker is not None:
    stage_table = compare_stages(
        test_parts, run["val_threshold"], stack_threshold,
        os.path.join(OUT["figures"], "stage_comparison.png"))

## Extending the pipeline (cheat sheet)

* **New experiment** — change `CONFIG`, give it a new `experiment_name`,
  re-run this notebook. Compare runs by comparing their folders.
* **New model** — add a class with `forward(x)` and one
  `@register_model("name")` line in `pipeline.py`, then set
  `CONFIG["model"]` to that name.
* **Turn on temporal labels** — add further assessment dates to
  `CITY_REGISTRY` in `pipeline.py` (notebook 1 lists which dates exist in
  your data), re-run notebook 1, then set `label_temporal: True` here.
* **Turn on spatial smoothing** — set `enabled: True`; threshold selection
  and evaluation handle it automatically. Compare the per-region metrics
  with and without: expect it to help dense, heavily damaged cities and to
  hurt sparse ones.
* **Optical imagery** — add `"s2"` to `PREP["sensors"]` in `pipeline.py`,
  re-run notebook 1, then set `sentinel2: True` here.

Continue with **notebook 3** for the interactive map.